# LLM-Driven Structured Parsing of Thermal Complaints into PMV and BACnet Setpoint Deltas

## The Processing Pipeline

**[ Occupant Feedback ]**

> "It's freezing near the window, but the room feels stuffy."

```text
        │
        ▼
┌────────────────────────────────────────────────────────┐
│ 1. Semantic Extraction & Contextual Binding            │
│    • Thermal Sensation: Cold (-2)                      │
│    • Air Quality Perception: Stuffy / Low Ventilation  │
│    • Spatial Anchor: Perimeter Zone (Near Window)      │
└────────────────────────────────────────────────────────┘
        │
        ▼
┌────────────────────────────────────────────────────────┐
│ 2. Physics & Standard Mapping (ASHRAE 55 / PMV)        │
│    • Cold near window ➔ Radiance / Cold Draft issue    │
│    • Stuffy air       ➔ CO₂ > 1000 ppm or low VAV      │
│    • Action Needed    ➔ Increase supply air temp,      │
│                          raise VAV min flow rate        │
└────────────────────────────────────────────────────────┘
        │
        ▼
┌────────────────────────────────────────────────────────┐
│ 3. Multi-Occupant Conflict Arbitration                 │
│    • Aggregates votes across space                     │
│    • Applies role weighting (Teacher vs. Student)      │
│    • Computes Pareto-optimal setpoint adjustment       │
└────────────────────────────────────────────────────────┘
        │
        ▼
┌────────────────────────────────────────────────────────┐
│ 4. Control Execution & Closed-Loop Response            │
│    • Transmits BACnet commands to VAV / Reheat coils   │
│    • Responds to user: "Adjusted perimeter heating     │
│      and increased fresh air exchange."                │
└────────────────────────────────────────────────────────┘

### How the Pipeline Operates

1. **Schema Enforcement (`Pydantic`):**  
   The `StructuredComfortAnalysis` class enforces strict types, enum categories, and numeric bounds (e.g., `estimated_pmv_score` constrained between $-3.0$ and $+3.0$).

2. **PMV Variable Inference:**  
   The LLM translates contextual clues (e.g., *"taking a test"* $\rightarrow 1.0\text{ met}$; *"shivering near the window"* $\rightarrow \text{PMV} = -1.8$) into standard ASHRAE 55 variables.

3. **Multi-Variable BACnet Deltas:**  
   Rather than simply adjusting a single global thermostat setpoint, the agent returns offset vectors targeting distinct HVAC actuators:
   - `temp_setpoint_delta_c`: Adjusts ambient zone setpoint.
   - `oa_damper_delta_pct`: Increases fresh air ventilation to resolve stuffiness.
   - `vav_reheat_delta_pct`: Drives hydronic perimeter heating to offset radiant cold near windows.

4. **Occupant Feedback Loop:**  
   Returns an automated, empathetic explanation confirming that action was taken and providing a realistic time estimate for thermal stabilization.

In [1]:
import json
from enum import Enum
from typing import Optional, List
from pydantic import BaseModel, Field

In [2]:
# =====================================================================
# 1. PYDANTIC SCHEMAS FOR STRUCTURED LLM OUTPUT
# =====================================================================

class ThermalSensation(str, Enum):
    HOT = "HOT"                     # PMV ~ +3.0
    WARM = "WARM"                   # PMV ~ +2.0
    SLIGHTLY_WARM = "SLIGHTLY_WARM" # PMV ~ +1.0
    NEUTRAL = "NEUTRAL"             # PMV ~ 0.0
    SLIGHTLY_COOL = "SLIGHTLY_COOL" # PMV ~ -1.0
    COOL = "COOL"                   # PMV ~ -2.0
    COLD = "COLD"                   # PMV ~ -3.0


class AirQualityPerception(str, Enum):
    STUFFY = "STUFFY"               # High CO2 / Low Fresh Air
    DRAFTY = "DRAFTY"               # High Local Air Velocity
    DRY = "DRY"                     # Low Relative Humidity (< 30%)
    HUMID = "HUMID"                 # High Relative Humidity (> 60%)
    NORMAL = "NORMAL"


class ExtractedPMVVariables(BaseModel):
    """Parameters mapping directly to ASHRAE Standard 55 / ISO 7730 PMV inputs."""
    metabolic_rate_met: float = Field(
        ..., 
        description="Estimated activity level in 'met' (1.0 = seated/testing, 1.4 = standing/lab, 2.0 = active sports)"
    )
    clothing_insulation_clo: float = Field(
        ..., 
        description="Estimated clothing insulation in 'clo' (0.5 = summer wear, 1.0 = winter suit/jacket)"
    )
    perceived_air_speed_m_s: float = Field(
        ..., 
        description="Estimated air velocity experienced by occupant in m/s (0.1 = still air, 0.5 = drafty)"
    )
    estimated_pmv_score: float = Field(
        ..., 
        ge=-3.0, le=3.0,
        description="Derived PMV index score ranging from -3.0 (cold) to +3.0 (hot)"
    )


class BACnetSetpointDeltas(BaseModel):
    """Calculated hardware adjustments to apply to BACnet controllers."""
    temp_setpoint_delta_c: float = Field(
        ..., 
        description="Temperature setpoint offset in °C (e.g., -0.5, +1.0)"
    )
    oa_damper_delta_pct: float = Field(
        ..., 
        description="Outdoor air fresh damper position change in % (e.g., +15.0 to resolve stuffiness)"
    )
    vav_reheat_delta_pct: float = Field(
        ..., 
        description="Hydronic reheat valve change in % (e.g., +25.0 to resolve cold perimeter draft)"
    )
    supply_fan_speed_delta_pct: float = Field(
        ..., 
        description="VAV fan speed change in % (e.g., -10.0 to reduce high draft speed)"
    )


class StructuredComfortAnalysis(BaseModel):
    """Complete structured payload returned by the LLM Agent."""
    target_zone: str = Field(..., description="Zone or room identifier parsed from feedback")
    sub_zone_location: Optional[str] = Field(None, description="Micro-zone context (e.g., 'window_row', 'podium')")
    thermal_sensation: ThermalSensation
    air_quality_perception: AirQualityPerception
    pmv_variables: ExtractedPMVVariables
    bacnet_deltas: BACnetSetpointDeltas
    occupant_response_msg: str = Field(
        ..., 
        description="Empathetic, clear explanation sent back to the teacher or student"
    )



In [3]:
# =====================================================================
# 2. SIMULATED LLM STRUCTURED OUTPUT PARSER
# =====================================================================

def parse_complaint_with_llm(
    natural_language_feedback: str, 
    zone_context: dict
) -> StructuredComfortAnalysis:
    """
    Simulates calling an LLM (such as OpenAI gpt-4o or Claude 3.5 Sonnet) 
    using Structured Outputs / Pydantic schema enforcement.
    """
    
    # SYSTEM PROMPT INSTRUCTING THE MODEL:
    # "You are an expert Building Services Engineer. Parse occupant comfort complaints, 
    #  infer ASHRAE 55 PMV parameters, calculate BACnet setpoint deltas, and write 
    #  a natural language feedback message."
    
    # Mocking the JSON response generated by the LLM matching StructuredComfortAnalysis schema
    mock_llm_json_response = {
        "target_zone": zone_context.get("zone_id", "Room_102"),
        "sub_zone_location": "perimeter_window_row",
        "thermal_sensation": "COLD",
        "air_quality_perception": "STUFFY",
        "pmv_variables": {
            "metabolic_rate_met": 1.0,         # Seated taking an exam
            "clothing_insulation_clo": 0.6,     # Light indoor clothing
            "perceived_air_speed_m_s": 0.12,   # Still air
            "estimated_pmv_score": -1.8        # Cold discomfort score
        },
        "bacnet_deltas": {
            "temp_setpoint_delta_c": 1.0,       # Raise target temp by +1.0°C
            "oa_damper_delta_pct": 20.0,        # Boost fresh air by +20% to clear stuffiness
            "vav_reheat_delta_pct": 35.0,       # Engage perimeter reheat coil
            "supply_fan_speed_delta_pct": 0.0
        },
        "occupant_response_msg": (
            "Thank you for reporting! We detected cold radiant conditions near the windows "
            "along with high stuffiness. We have increased perimeter heating by 1.0°C "
            "and boosted fresh air exchange by 20%. Conditions should normalize in 6–8 minutes."
        )
    }

    # Validate and parse into Pydantic instance
    return StructuredComfortAnalysis(**mock_llm_json_response)

In [4]:
# =====================================================================
# 3. BACNET DRIVER WRAPPER (Applying Deltas to Hardware)
# =====================================================================

class BACnetSetpointAdapter:
    """Applies calculated deltas to active BACnet Present_Value properties."""
    
    def apply_deltas(self, zone_id: str, deltas: BACnetSetpointDeltas):
        print(f"\n--- [BACnet Direct Override Executed for {zone_id}] ---")
        print(f"  ├── Temp Setpoint Adj : +{deltas.temp_setpoint_delta_c}°C")
        print(f"  ├── Outdoor Air Damper: +{deltas.oa_damper_delta_pct}% (Fresh Air Pulse)")
        print(f"  ├── Reheat Valve      : +{deltas.vav_reheat_delta_pct}% (Perimeter Warmth)")
        print(f"  └── Fan Speed Adj     : {deltas.supply_fan_speed_delta_pct}%")


In [5]:
# =====================================================================
# 4. EXECUTION PIPELINE
# =====================================================================

if __name__ == "__main__":
    # Sample Input: Student/Teacher Natural Language Complaint
    complaint_text = (
        "Students near the windows in Room 102 are shivering during their test, "
        "but the middle of the room feels really stuffy and heavy."
    )
    
    current_zone_context = {
        "zone_id": "Classroom_102",
        "current_temp": 20.5,
        "current_co2": 1150
    }
    
    print(f"User Complaint: \"{complaint_text}\"\n")
    
    # Step 1: Parse complaint into structured PMV & BACnet parameters
    analysis = parse_complaint_with_llm(complaint_text, current_zone_context)
    
    # Step 2: Inspect Structured JSON Output
    print("=== Extracted PMV & Comfort Parameters ===")
    print(f"Primary Sensation : {analysis.thermal_sensation.value}")
    print(f"Air Perception    : {analysis.air_quality_perception.value}")
    print(f"Derived PMV Score : {analysis.pmv_variables.estimated_pmv_score}")
    print(f"Activity (Met)    : {analysis.pmv_variables.metabolic_rate_met} met")
    print(f"Clothing (Clo)    : {analysis.pmv_variables.clothing_insulation_clo} clo")
    
    # Step 3: Apply Deltas to BACnet
    bacnet_adapter = BACnetSetpointAdapter()
    bacnet_adapter.apply_deltas(analysis.target_zone, analysis.bacnet_deltas)
    
    # Step 4: Return Natural Language Message to Occupant
    print("\n=== Closed-Loop Occupant Feedback ===")
    print(f"Response to App: \"{analysis.occupant_response_msg}\"")

User Complaint: "Students near the windows in Room 102 are shivering during their test, but the middle of the room feels really stuffy and heavy."

=== Extracted PMV & Comfort Parameters ===
Primary Sensation : COLD
Air Perception    : STUFFY
Derived PMV Score : -1.8
Activity (Met)    : 1.0 met
Clothing (Clo)    : 0.6 clo

--- [BACnet Direct Override Executed for Classroom_102] ---
  ├── Temp Setpoint Adj : +1.0°C
  ├── Outdoor Air Damper: +20.0% (Fresh Air Pulse)
  ├── Reheat Valve      : +35.0% (Perimeter Warmth)
  └── Fan Speed Adj     : 0.0%

=== Closed-Loop Occupant Feedback ===
Response to App: "Thank you for reporting! We detected cold radiant conditions near the windows along with high stuffiness. We have increased perimeter heating by 1.0°C and boosted fresh air exchange by 20%. Conditions should normalize in 6–8 minutes."
